In [13]:
# ==============================================
# Imports & Setup
# ==============================================
from pathlib import Path
import sys
import importlib
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tabulate import tabulate

# Parent folder hinzufügen
sys.path.append(str(Path("..").resolve()))

import proximitylib.proximity as prox
importlib.reload(prox)


param_choice = 3
params = prox.param_sets[param_choice]

# ==============================================
# Tabelle plotten (matplotlib)
# ==============================================
def plot_summary_table(ax, df, title):
    ax.axis("off")

    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        rowLabels=df.index,
        loc="center"
    )

    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.6)

    ax.set_title(title, fontsize=12, fontweight="bold", pad=10)


# ==============================================
# Einzelne Peak-Matrix plotten
# ==============================================
def plot_peak_matrix(ax, path, params):
    df = pd.read_csv(path)
    iois = df["IOI"].dropna().values
    n = len(iois)

    df_pairs, _, _ = prox.build_pairwise_proximity_df(iois, params)
    pivot_vals = df_pairs.pivot(index="i", columns="j", values="prox_value") * 100
    pivot_labels = df_pairs.pivot(index="i", columns="j", values="peak_label")

    # Farben (wie im Original)
    tab10 = plt.cm.tab10.colors
    fixed_colors = {
        "1/1": tab10[0], "2/1": tab10[1], "3/1": tab10[2],
        "4/1": tab10[3], "5/1": tab10[4],
        "1/2": tab10[5], "3/2": tab10[6],
        "5/2": tab10[7], "1/3": tab10[8]
    }

    for (i, j), val in np.ndenumerate(pivot_vals.values):
        label = pivot_labels.iloc[i, j]
        base_color = fixed_colors.get(label, (0.7, 0.7, 0.7))
        alpha = val / 100 if not np.isnan(val) else 0

        ax.add_patch(
            plt.Rectangle(
                (j, i), 1, 1,
                facecolor=(*base_color, alpha),
                edgecolor="none"
            )
        )

    ax.set_xlim(0, n)
    ax.set_ylim(0, n)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.axis("off")
    ax.set_title(Path(path).stem, fontsize=9, fontweight="bold")


# ==============================================
# Analyse (Original show_results, aber ohne Print)
# ==============================================
def get_summary_table(path, name, params, num_null=1000, alpha=0.1):
    df, distance_corr_dict = prox.analyze_dataset_peaks_table(
        path, params,
        num_null=num_null,
        alpha=alpha
    )

    df_summary = prox.summarize_results_table(
        df,
        distance_corr_dict=distance_corr_dict,
        alpha=alpha
    )

    return df_summary


# ==============================================
# INPUT
# ==============================================
matrix_files = [
    "../data/Seba's short-tailed bat (Carollia perspicillata)/C_persp_IC_p4_b4.csv",
    "../data/Seba's short-tailed bat (Carollia perspicillata)/C_persp_IC_p4_b9.csv",
    "../data/Seba's short-tailed bat (Carollia perspicillata)/C_persp_IC_p2_b7.csv",
    "../data/Seba's short-tailed bat (Carollia perspicillata)/C_persp_IC_p3_b2.csv",
    "../data/Seba's short-tailed bat (Carollia perspicillata)/C_persp_IC_p1_b3.csv",
]

summary_dataset_path = "../data/Seba's short-tailed bat (Carollia perspicillata)"
summary_name = "Carollia perspicillata"

# ==============================================
# Analyse-Tabelle holen
# ==============================================
df_summary = get_summary_table(
    summary_dataset_path,
    summary_name,
    params
)

# ==============================================
# FIGURE LAYOUT
# ==============================================
fig = plt.figure(figsize=(22, 10))
gs = gridspec.GridSpec(
    nrows=2,
    ncols=5,
    height_ratios=[1.2, 1],
    hspace=0.35
)

# ==============================================
# MATRIZEN (oben)
# ==============================================
for i, path in enumerate(matrix_files):
    ax = fig.add_subplot(gs[0, i])
    plot_peak_matrix(ax, path, params)

# ==============================================
# TABELLE (unten, über volle Breite)
# ==============================================
ax_table = fig.add_subplot(gs[1, :])
plot_summary_table(
    ax_table,
    df_summary,
    title="Summary Results - Carollia perspicillata"
)

plt.show()


KeyboardInterrupt: 